In [1]:
from pathlib import Path
import pandas as pd

In [2]:
DATA = Path(r"D:\amazon\Import Data")

In [3]:
files = {
    "customer": "Customer.csv",
    "delivery": "Delievry.csv",
    "orders": "Orders.csv",
    "products": "Products.csv",
    "ratings": "Ratings.csv",
    "returns": "Returns_refund.csv",
    "subscription_details": "Subscription_Details.csv",
    "subscription_plan": "Subscription_Plan.csv",
    "transactions": "Transactions.csv",
}

In [4]:
raw = {
    name: pd.read_csv(DATA / filename, dtype="string")
    for name, filename in files.items()
}

In [5]:
for name, df in raw.items():
    print(f"\n{name}: {df.shape[0]:,} rows × {df.shape[1]} columns")
    print("Columns:", df.columns.tolist())
    print("Missing values:")
    print(df.isna().sum())
    print("Exact duplicate rows:", df.duplicated().sum())


customer: 10,184 rows × 8 columns
Columns: ['C_ID', 'C_Name', 'Gender', 'Age', 'City', 'State', 'Street Address', 'Mobile']
Missing values:
C_ID              0
C_Name            0
Gender            0
Age               0
City              0
State             0
Street Address    0
Mobile            0
dtype: int64
Exact duplicate rows: 184

delivery: 325 rows × 3 columns
Columns: ['DP_ID', 'DP_Name', 'DP_Ratings']
Missing values:
DP_ID          0
DP_Name        0
DP_Ratings    22
dtype: int64
Exact duplicate rows: 25

orders: 10,000 rows × 8 columns
Columns: ['Or_ID', 'C_ID', 'P_ID', 'Order_Date', 'Order_Time', 'Qty', 'Coupon', 'DP_ID']
Missing values:
Or_ID            0
C_ID             0
P_ID             0
Order_Date       0
Order_Time       0
Qty              0
Coupon        5011
DP_ID            0
dtype: int64
Exact duplicate rows: 0

products: 10,000 rows × 5 columns
Columns: ['P_ID', 'PName', 'Category', 'Specs1', 'Price']
Missing values:
P_ID         0
PName       14
Category     

In [6]:
for name, key in [("customer", "C_ID"), ("delivery", "DP_ID")]:
    df = raw[name]
    repeated = df[df.duplicated(key, keep=False)].sort_values(key)

    print(f"\n{name}: rows with repeated {key} =", len(repeated))
    print(repeated.head(10).to_string(index=False))


customer: rows with repeated C_ID = 368
       C_ID           C_Name Gender Age          City          State                                         Street Address     Mobile
CS_11000008 Darsh Brahmbhatt Female  32 Visakhapatnam Andhra Pradesh Main St, 481 , Visakhapatnam , Andhra Pradesh - 177603 9132998537
CS_11000008 Darsh Brahmbhatt Female  32 Visakhapatnam Andhra Pradesh Main St, 481 , Visakhapatnam , Andhra Pradesh - 177603 9132998537
CS_11000009     Radha Kurian   Male  63     Ghaziabad  Uttar Pradesh       3rd St, 327 , Ghaziabad , Uttar Pradesh - 024657 9877455959
CS_11000009     Radha Kurian   Male  63     Ghaziabad  Uttar Pradesh       3rd St, 327 , Ghaziabad , Uttar Pradesh - 024657 9877455959
CS_11000010        Aryan Oak Female  70       Kolkata    West Bengal          Pine St, 495 , Kolkata , West Bengal - 700353 9321097454
CS_11000010        Aryan Oak Female  70       Kolkata    West Bengal          Pine St, 495 , Kolkata , West Bengal - 700353 9321097454
CS_11000011   

In [7]:
for name, key in [("customer", "C_ID"), ("delivery", "DP_ID")]:
    df = raw[name]
    repeated = df[df.duplicated(key, keep=False)]

    # Count distinct versions of each repeated ID across all columns
    versions = repeated.groupby(key, dropna=False).nunique()
    conflicts = versions[(versions > 1).any(axis=1)]

    print(f"{name}: conflicting IDs =", len(conflicts))

customer: conflicting IDs = 0
delivery: conflicting IDs = 0


In [8]:
clean = {name: df.copy() for name, df in raw.items()}

clean["customer"] = clean["customer"].drop_duplicates()
clean["delivery"] = clean["delivery"].drop_duplicates()

print("Customer:", clean["customer"].shape)
print("Delivery:", clean["delivery"].shape)
print("Customer IDs unique:", clean["customer"]["C_ID"].is_unique)
print("Delivery IDs unique:", clean["delivery"]["DP_ID"].is_unique)

Customer: (10000, 8)
Delivery: (300, 3)
Customer IDs unique: True
Delivery IDs unique: True


In [9]:
primary_keys = {
    "orders": "Or_ID",
    "products": "P_ID",
    "transactions": "Tr_ID",
    "ratings": "R_ID",
    "returns": "RT_ID",
    "subscription_details": "SD_ID",
    "subscription_plan": "Plan_ID",
}

for table, key in primary_keys.items():
    df = clean[table]
    duplicate_count = df[key].duplicated().sum()
    print(f"{table}: duplicate {key} values = {duplicate_count}")

orders: duplicate Or_ID values = 0
products: duplicate P_ID values = 0
transactions: duplicate Tr_ID values = 0
ratings: duplicate R_ID values = 0
returns: duplicate RT_ID values = 0
subscription_details: duplicate SD_ID values = 0
subscription_plan: duplicate Plan_ID values = 0


In [10]:
relationships = [
    ("orders", "C_ID", "customer", "C_ID"),
    ("orders", "P_ID", "products", "P_ID"),
    ("orders", "DP_ID", "delivery", "DP_ID"),
    ("transactions", "Or_ID", "orders", "Or_ID"),
    ("ratings", "Or_ID", "orders", "Or_ID"),
    ("returns", "Or_ID", "orders", "Or_ID"),
    ("subscription_details", "C_ID", "customer", "C_ID"),
    ("subscription_details", "Plan_ID", "subscription_plan", "Plan_ID"),
]

for child_table, child_col, parent_table, parent_col in relationships:
    child_values = set(clean[child_table][child_col].dropna())
    parent_values = set(clean[parent_table][parent_col].dropna())
    orphan_values = child_values - parent_values

    print(
        f"{child_table}.{child_col} → {parent_table}.{parent_col}: "
        f"{len(orphan_values)} orphan IDs"
    )

orders.C_ID → customer.C_ID: 0 orphan IDs
orders.P_ID → products.P_ID: 0 orphan IDs
orders.DP_ID → delivery.DP_ID: 0 orphan IDs
transactions.Or_ID → orders.Or_ID: 0 orphan IDs
ratings.Or_ID → orders.Or_ID: 0 orphan IDs
returns.Or_ID → orders.Or_ID: 0 orphan IDs
subscription_details.C_ID → customer.C_ID: 0 orphan IDs
subscription_details.Plan_ID → subscription_plan.Plan_ID: 0 orphan IDs


In [11]:
# Convert numeric columns
numeric_columns = {
    "customer": ["Age"],
    "products": ["Price"],
    "orders": ["Qty"],
    "ratings": ["Prod_Rating", "Delivery/Service_Rating"],
    "delivery": ["DP_Ratings"],
}

for table, columns in numeric_columns.items():
    for col in columns:
        original = clean[table][col]
        converted = pd.to_numeric(original, errors="coerce")
        failed = original.notna() & converted.isna()
        print(f"{table}.{col}: conversion failures = {failed.sum()}")
        clean[table][col] = converted

# Convert date columns
date_columns = {
    "orders": ["Order_Date"],
    "returns": ["Date"],
    "subscription_details": ["From_Date", "To_Date"],
}

for table, columns in date_columns.items():
    for col in columns:
        original = clean[table][col]
        converted = pd.to_datetime(original, errors="coerce")
        failed = original.notna() & converted.isna()
        print(f"{table}.{col}: conversion failures = {failed.sum()}")
        clean[table][col] = converted

customer.Age: conversion failures = 0
products.Price: conversion failures = 0
orders.Qty: conversion failures = 0
ratings.Prod_Rating: conversion failures = 0
ratings.Delivery/Service_Rating: conversion failures = 0
delivery.DP_Ratings: conversion failures = 0
orders.Order_Date: conversion failures = 0
returns.Date: conversion failures = 0
subscription_details.From_Date: conversion failures = 0
subscription_details.To_Date: conversion failures = 0


In [12]:
print("Age outside 18–100:", (~clean["customer"]["Age"].between(18, 100)).sum())
print("Non-positive prices:", (clean["products"]["Price"] <= 0).sum())
print("Non-positive quantities:", (clean["orders"]["Qty"] <= 0).sum())

for col in ["Prod_Rating", "Delivery/Service_Rating"]:
    bad = clean["ratings"][col].notna() & ~clean["ratings"][col].between(1, 5)
    print(f"Ratings.{col} outside 1–5:", bad.sum())

bad_subscription_dates = (
    clean["subscription_details"]["To_Date"]
    < clean["subscription_details"]["From_Date"]
)
print("Subscription end date before start date:", bad_subscription_dates.sum())

Age outside 18–100: 0
Non-positive prices: 0
Non-positive quantities: 0
Ratings.Prod_Rating outside 1–5: 0
Ratings.Delivery/Service_Rating outside 1–5: 0
Subscription end date before start date: 1177


In [13]:
sd = clean["subscription_details"].copy()

bad_dates = sd[sd["To_Date"] < sd["From_Date"]]

print("Rows:", len(bad_dates))
print("\nBy plan:")
print(bad_dates["Plan_ID"].value_counts(dropna=False))

print("\nExamples:")
display(bad_dates.head(10))

print("\nDate span:")
print("From_Date:", bad_dates["From_Date"].min(), "to", bad_dates["From_Date"].max())
print("To_Date:  ", bad_dates["To_Date"].min(), "to", bad_dates["To_Date"].max())

Rows: 1177

By plan:
Plan_ID
Pln_51000001    410
Pln_51000003    387
Pln_51000002    380
Name: count, dtype: int64[pyarrow]

Examples:


,SD_ID,C_ID,Plan_ID,From_Date,To_Date
8,SPD_201000009,CS_11007630,Pln_51000002,2024-12-06,2024-02-17
13,SPD_201000014,CS_11005667,Pln_51000001,2024-10-08,2024-09-18
16,SPD_201000017,CS_11005555,Pln_51000003,2024-05-05,2024-04-18
20,SPD_201000021,CS_11009963,Pln_51000003,2024-12-18,2024-10-09
22,SPD_201000023,CS_11008755,Pln_51000001,2024-08-16,2024-05-06
25,SPD_201000026,CS_11004229,Pln_51000002,2024-08-07,2024-01-26
30,SPD_201000031,CS_11003649,Pln_51000002,2024-01-09,2024-01-05
42,SPD_201000043,CS_11003776,Pln_51000002,2024-04-15,2024-03-30
48,SPD_201000049,CS_11000591,Pln_51000002,2024-06-30,2024-06-10
63,SPD_201000064,CS_11000778,Pln_51000002,2024-09-15,2024-01-18



Date span:
From_Date: 2024-01-03 00:00:00 to 2024-12-31 00:00:00
To_Date:   2024-01-01 00:00:00 to 2024-12-20 00:00:00


In [14]:
for name, df in clean.items():
    missing = df.isna().sum()
    missing = missing[missing > 0]

    print(f"\n{name}:")
    print(missing if not missing.empty else "No missing values")


customer:
No missing values

delivery:
DP_Ratings    20
dtype: int64

orders:
Coupon    5011
dtype: int64

products:
PName    14
dtype: int64

ratings:
Prod_Rating                22
Delivery/Service_Rating    29
dtype: int64

returns:
No missing values

subscription_details:
No missing values

subscription_plan:
No missing values

transactions:
Transaction_Mode    33
dtype: int64


In [15]:
clean["orders"]["Coupon"] = clean["orders"]["Coupon"].fillna("No Coupon")
clean["products"]["PName"] = clean["products"]["PName"].fillna("Unknown Product")
clean["transactions"]["Transaction_Mode"] = (
    clean["transactions"]["Transaction_Mode"].fillna("Unknown")
)

# Leave these numeric ratings as missing:
# clean["ratings"]["Prod_Rating"]
# clean["ratings"]["Delivery/Service_Rating"]
# clean["delivery"]["DP_Ratings"]

for name, df in clean.items():
    print(f"\n{name}:")
    print(df.isna().sum()[lambda s: s > 0])


customer:
Series([], dtype: int64)

delivery:
DP_Ratings    20
dtype: int64

orders:
Series([], dtype: int64)

products:
Series([], dtype: int64)

ratings:
Prod_Rating                22
Delivery/Service_Rating    29
dtype: int64

returns:
Series([], dtype: int64)

subscription_details:
Series([], dtype: int64)

subscription_plan:
Series([], dtype: int64)

transactions:
Series([], dtype: int64)


EXPLORATORY DATA ANALYSIS

In [16]:
orders_eda = (
    clean["orders"]
    .merge(
        clean["products"][["P_ID", "PName", "Category", "Price"]],
        on="P_ID",
        how="left",
        validate="many_to_one"
    )
    .merge(
        clean["customer"][["C_ID", "Gender", "Age", "City", "State"]],
        on="C_ID",
        how="left",
        validate="many_to_one"
    )
)

orders_eda["Gross_Sales_Proxy"] = orders_eda["Qty"] * orders_eda["Price"]
orders_eda["Month"] = orders_eda["Order_Date"].dt.to_period("M").dt.to_timestamp()

print("Rows:", len(orders_eda))
print("Distinct orders:", orders_eda["Or_ID"].nunique())
print("Missing product prices:", orders_eda["Price"].isna().sum())
print("Missing customers:", orders_eda["C_ID"].isna().sum())
print("Gross sales proxy:", round(orders_eda["Gross_Sales_Proxy"].sum(), 2))

display(orders_eda.head())

Rows: 10000
Distinct orders: 10000
Missing product prices: 0
Missing customers: 0
Gross sales proxy: 13987259.77


,Or_ID,C_ID,P_ID,Order_Date,Order_Time,Qty,Coupon,DP_ID,PName,Category,Price,Gender,Age,City,State,Gross_Sales_Proxy,Month
0,OR_31000001,CS_11005317,PD_21001301,2024-02-27,22:02:00,1,No Coupon,DV_61000087,Mission,Fitness,330.13,Male,67,Mumbai,Maharashtra,330.13,2024-02-01
1,OR_31000002,CS_11000423,PD_21003593,2024-01-21,08:33:31,5,PULL,DV_61000085,What,Fashion,485.81,Female,59,Bengaluru,Karnataka,2429.05,2024-01-01
2,OR_31000003,CS_11001042,PD_21004315,2024-09-22,17:26:05,1,No Coupon,DV_61000237,Office,Books,115.58,Male,44,Delhi,Delhi,115.58,2024-09-01
3,OR_31000004,CS_11004079,PD_21007443,2023-05-26,03:15:48,10,AGREEMENT,DV_61000292,Sea,Home Appliance,234.48,Female,57,Ghaziabad,Uttar Pradesh,2344.8,2023-05-01
4,OR_31000005,CS_11009894,PD_21007621,2023-10-26,04:02:44,7,WINDOW,DV_61000096,Though,Electronics,124.24,Male,28,Visakhapatnam,Andhra Pradesh,869.68,2023-10-01


In [17]:
monthly_sales = (
    orders_eda
    .groupby("Month", as_index=False)
    .agg(
        Orders=("Or_ID", "nunique"),
        Units=("Qty", "sum"),
        Gross_Sales_Proxy=("Gross_Sales_Proxy", "sum")
    )
    .sort_values("Month")
)

monthly_sales["Gross_Sales_Proxy"] = monthly_sales["Gross_Sales_Proxy"].round(2)

display(monthly_sales)

,Month,Orders,Units,Gross_Sales_Proxy
0,2023-01-01,427,2472,628430.03
1,2023-02-01,409,2145,529528.52
2,2023-03-01,415,2364,573985.86
3,2023-04-01,407,2262,573670.74
4,2023-05-01,445,2463,633183.83
5,2023-06-01,403,2249,557366.75
6,2023-07-01,378,2071,518025.73
7,2023-08-01,403,2092,514171.31
8,2023-09-01,391,2137,568397.08
9,2023-10-01,420,2239,538597.3


In [18]:
category_sales = (
    orders_eda
    .groupby("Category", as_index=False)
    .agg(
        Orders=("Or_ID", "nunique"),
        Units=("Qty", "sum"),
        Gross_Sales_Proxy=("Gross_Sales_Proxy", "sum"),
        Products=("P_ID", "nunique")
    )
)

category_sales["Sales_Share_Pct"] = (
    category_sales["Gross_Sales_Proxy"]
    / category_sales["Gross_Sales_Proxy"].sum()
    * 100
).round(1)

category_sales["Gross_Sales_Proxy"] = category_sales["Gross_Sales_Proxy"].round(2)

display(
    category_sales
    .sort_values("Gross_Sales_Proxy", ascending=False)
    .reset_index(drop=True)
)

,Category,Orders,Units,Gross_Sales_Proxy,Products,Sales_Share_Pct
0,Electronics,2055,11450,2977529.58,1276,21.3
1,Fashion,2060,11344,2887254.62,1285,20.6
2,Home Appliance,1999,11077,2790930.93,1252,20.0
3,Books,1923,10524,2682161.67,1194,19.2
4,Fitness,1963,10741,2649382.97,1259,18.9


In [19]:
top_products = (
    orders_eda
    .groupby(["P_ID", "PName", "Category"], as_index=False)
    .agg(
        Orders=("Or_ID", "nunique"),
        Units=("Qty", "sum"),
        Gross_Sales_Proxy=("Gross_Sales_Proxy", "sum")
    )
    .sort_values("Gross_Sales_Proxy", ascending=False)
    .head(10)
)

top_products["Gross_Sales_Proxy"] = top_products["Gross_Sales_Proxy"].round(2)
display(top_products.reset_index(drop=True))

,P_ID,PName,Category,Orders,Units,Gross_Sales_Proxy
0,PD_21006446,Its,Books,5,37,16862.38
1,PD_21000844,Task,Home Appliance,5,33,15512.64
2,PD_21007243,Future,Fitness,4,35,14476.0
3,PD_21002819,Somebody,Electronics,6,34,14151.48
4,PD_21008237,Southern,Fashion,6,33,14060.31
5,PD_21009869,Seek,Home Appliance,4,37,13701.47
6,PD_21007446,Behind,Electronics,4,31,13480.35
7,PD_21001659,Its,Books,5,34,13331.74
8,PD_21000038,Near,Books,4,29,13079.87
9,PD_21003606,Former,Home Appliance,6,33,12818.52


In [20]:
state_sales = (
    orders_eda
    .groupby("State", as_index=False)
    .agg(
        Orders=("Or_ID", "nunique"),
        Customers=("C_ID", "nunique"),
        Units=("Qty", "sum"),
        Gross_Sales_Proxy=("Gross_Sales_Proxy", "sum")
    )
)

state_sales["Avg_Order_Value_Proxy"] = (
    state_sales["Gross_Sales_Proxy"] / state_sales["Orders"]
).round(2)

state_sales["Sales_Share_Pct"] = (
    state_sales["Gross_Sales_Proxy"]
    / state_sales["Gross_Sales_Proxy"].sum()
    * 100
).round(1)

state_sales["Gross_Sales_Proxy"] = state_sales["Gross_Sales_Proxy"].round(2)

display(
    state_sales
    .sort_values("Gross_Sales_Proxy", ascending=False)
    .reset_index(drop=True)
)

,State,Orders,Customers,Units,Gross_Sales_Proxy,Avg_Order_Value_Proxy,Sales_Share_Pct
0,Maharashtra,1994,1239,10972,2788692.4,1398.54,19.9
1,Gujarat,1551,989,8533,2160971.01,1393.28,15.4
2,Uttar Pradesh,1487,951,8066,2073340.3,1394.31,14.8
3,Madhya Pradesh,977,622,5358,1345349.19,1377.02,9.6
4,Andhra Pradesh,554,334,3176,784175.84,1415.48,5.6
5,Tamil Nadu,541,331,2979,742219.64,1371.94,5.3
6,Bihar,510,326,2861,726778.45,1425.06,5.2
7,Karnataka,478,293,2600,706016.61,1477.02,5.0
8,West Bengal,490,309,2798,702706.29,1434.09,5.0
9,Telangana,513,341,2874,699324.2,1363.21,5.0


In [21]:
payment_summary = (
    clean["transactions"]
    .groupby(["Transaction_Mode", "Tran_Status"], as_index=False)
    .agg(
        Transaction_Attempts=("Tr_ID", "count"),
        Distinct_Orders=("Or_ID", "nunique")
    )
    .sort_values(["Transaction_Mode", "Tran_Status"])
)

display(payment_summary)

,Transaction_Mode,Tran_Status,Transaction_Attempts,Distinct_Orders
0,Credit Card,Failed,991,953
1,Credit Card,Successful,989,941
2,Debit Card,Failed,996,954
3,Debit Card,Successful,1015,964
4,Net Banking,Failed,1019,960
5,Net Banking,Successful,1005,960
6,UPI,Failed,1002,960
7,UPI,Successful,976,922
8,Unknown,Failed,13,13
9,Unknown,Successful,20,20


In [22]:
returns_summary = (
    clean["returns"]
    .groupby(["Return/Refund", "Reason"], as_index=False)
    .agg(
        Return_Records=("RT_ID", "count"),
        Affected_Orders=("Or_ID", "nunique")
    )
    .sort_values(["Return/Refund", "Return_Records"], ascending=[True, False])
)

display(returns_summary)

affected_orders = clean["returns"]["Or_ID"].nunique()
total_orders = clean["orders"]["Or_ID"].nunique()

print(f"Orders with at least one return/refund: {affected_orders:,} of {total_orders:,}")
print(f"Order incidence: {affected_orders / total_orders:.1%}")

,Return/Refund,Reason,Return_Records,Affected_Orders
1,Refund,Late Delivery,352,351
0,Refund,Defective Product,342,337
2,Refund,Wrong Item Shipped,337,330
5,Return,Wrong Item Shipped,331,328
3,Return,Defective Product,328,324
4,Return,Late Delivery,310,306


Orders with at least one return/refund: 1,820 of 10,000
Order incidence: 18.2%


In [23]:
rating_cols = ["Prod_Rating", "Delivery/Service_Rating"]

rating_summary = clean["ratings"][rating_cols].agg(["count", "mean", "median"]).T
display(rating_summary.round(2))

for col in rating_cols:
    print(f"\n{col} response counts:")
    print(clean["ratings"][col].value_counts(dropna=False).sort_index())

,count,mean,median
Prod_Rating,9978.0,2.99,3.0
Delivery/Service_Rating,9971.0,3.01,3.0



Prod_Rating response counts:
Prod_Rating
1.0     1988
2.0     2003
3.0     2030
4.0     2012
5.0     1945
<NA>      22
Name: count, dtype: Int64

Delivery/Service_Rating response counts:
Delivery/Service_Rating
1.0     1960
2.0     2000
3.0     1936
4.0     2089
5.0     1986
<NA>      29
Name: count, dtype: Int64


In [24]:
subs = clean["subscription_details"].copy()

# Ensure the invalid-range flag exists
subs["Invalid_Date_Range"] = subs["To_Date"] < subs["From_Date"]

valid_subs = subs.loc[~subs["Invalid_Date_Range"]].copy()
valid_subs["Duration_Days"] = (
    valid_subs["To_Date"] - valid_subs["From_Date"]
).dt.days

# Use the last date in the order data as the reporting snapshot
as_of = clean["orders"]["Order_Date"].max()

valid_subs["Active_As_Of_Snapshot"] = (
    (valid_subs["From_Date"] <= as_of) &
    (valid_subs["To_Date"] >= as_of)
)

subscription_summary = (
    valid_subs
    .merge(
        clean["subscription_plan"][["Plan_ID", "Plan_Name"]],
        on="Plan_ID",
        how="left",
        validate="many_to_one"
    )
    .groupby(["Plan_ID", "Plan_Name"], as_index=False)
    .agg(
        Subscriptions=("SD_ID", "count"),
        Customers=("C_ID", "nunique"),
        Median_Duration_Days=("Duration_Days", "median"),
        Active_As_Of_Snapshot=("Active_As_Of_Snapshot", "sum")
    )
)

print("Reporting snapshot:", as_of.date())
print("Valid subscriptions:", len(valid_subs))
print("Excluded invalid date ranges:", int(subs["Invalid_Date_Range"].sum()))
display(subscription_summary)

Reporting snapshot: 2024-12-31
Valid subscriptions: 8859
Excluded invalid date ranges: 1177


,Plan_ID,Plan_Name,Subscriptions,Customers,Median_Duration_Days,Active_As_Of_Snapshot
0,Pln_51000001,Prime Basic,2950,2530,419.0,1714
1,Pln_51000002,Prime Standard,3013,2604,420.0,1697
2,Pln_51000003,Prime Premium,2896,2509,416.5,1625
